In [ ]:
import importlib.util
if importlib.util.find_spec("spytial") is None:
    try:
        import piplite
        await piplite.install("spytial-diagramming")
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spytial-diagramming"])
from spytial import *
import spytial.utils as _su
if not _su.is_notebook():
    _su.is_notebook = lambda: True  # JupyterLite/Pyodide: render diagram() inline
from spytial.annotations import *

# Incremental DAG construction: step-by-step

Build a DAG one edge at a time. Each frame adds a single edge — every
intermediate state is still a sub-DAG, so the same
`orientation(..., directions=["right"])` rule keeps the partial graph
topologically laid out left-to-right at every step.

This is `stability` territory: the vertices that already exist stay
where they are; only one new edge appears per frame.

In [ ]:
dag_adj_matrix = [
    [0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0],
]

ADJ_MATRIX_TO_EDGE_SELECTOR = "{i, j: int | 1 in (list.idx[i]).idx[j]}"

## Demo

Mutate a shared adjacency matrix in place, recording a frame after each edge insertion.

In [ ]:
n = len(dag_adj_matrix)
partial = [[0] * n for _ in range(n)]

# Decorate ONCE; the same list object is mutated and re-recorded each frame.
partial = inferredEdge(selector=ADJ_MATRIX_TO_EDGE_SELECTOR, name="")(partial)
partial = orientation(selector=ADJ_MATRIX_TO_EDGE_SELECTOR, directions=["right"])(partial)
partial = hideAtom(selector="list")(partial)

seq = sequence(
    sequence_policy="stability",
    title="Building the DAG one edge at a time",
)

seq.record(partial, label="START: empty DAG")
for i in range(n):
    for j in range(n):
        if dag_adj_matrix[i][j]:
            partial[i][j] = 1
            seq.record(partial, label=f"add edge {i} → {j}")

seq.diagram(method="browser")